# Moving Average Trading Strategy Backtest

This project analyzes stock data using a moving average crossover strategy and compares it to a buy-and-hold approach.

## Import Libraries

We import the required libraries for data collection and visualization.

In [1]:
import yfinance as yf
import matplotlib.pyplot as plt

## Data Collection

We download historical stock price data using yfinance.
The user can input a stock ticker and time period.

In [ ]:
ticker = input("Enter stock ticker (e.g. AAPL, TSLA, NVDA): ")

start_date = input("Enter start date (YYYY-MM-DD): ")
end_date = input("Enter end date (YYYY-MM-DD): ")

data = yf.download(ticker, start=start_date, end=end_date)

## Data Preparation

We extract the closing prices and convert them into a list for analysis.

In [ ]:
prices = data["Close"].squeeze().tolist()

In [ ]:
short_window = 5
long_window = 10

## Strategy

We implement a moving average crossover strategy:

- Buy when short-term average > long-term average  
- Stay out of the market otherwise

In [ ]:
signals = []
positions = []
daily_returns = []
strategy_returns = []

position = 0

for i in range(len(prices)):

    # daily return
    if i == 0:
        day_return = 0
    else:
        day_return = (prices[i] - prices[i-1]) / prices[i-1]
    daily_returns.append(day_return)

    # not enough data yet
    if i < long_window - 1:
        signals.append("No signal")
        positions.append(position)
        strategy_returns.append(0)
    else:
        short_ma = sum(prices[i-short_window+1:i+1]) / short_window
        long_ma = sum(prices[i-long_window+1:i+1]) / long_window

        if short_ma > long_ma:
            signal = "Buy"
            position = 1
        else:
            signal = "Sell"
            position = 0

        signals.append(signal)
        positions.append(position)

        strat_return = positions[i-1] * day_return
        strategy_returns.append(strat_return)

In [ ]:
strategy_equity = []
buy_hold_equity = []

strategy_value = 1
buy_hold_value = 1

for i in range(len(prices)):
    strategy_value *= (1 + strategy_returns[i])
    buy_hold_value *= (1 + daily_returns[i])

    strategy_equity.append(strategy_value)
    buy_hold_equity.append(buy_hold_value)

In [ ]:
strategy_total_return = strategy_equity[-1] - 1
buy_hold_return = buy_hold_equity[-1] - 1

print(f"\n--- {ticker} Results ({start_date} to {end_date}) ---")
print(f"Strategy Return: {round(strategy_total_return, 4)}")
print(f"Buy & Hold Return: {round(buy_hold_return, 4)}")

## Price Visualization

This graph shows how the stock price changes over time.

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(prices, label="Price")
plt.title(f"{ticker} Price")
plt.legend()
plt.show()

## Strategy Performance

We compare the performance of the strategy with a buy-and-hold approach using an equity curve.

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(strategy_equity, label="Strategy")
plt.plot(buy_hold_equity, label="Buy & Hold")
plt.title(f"{ticker} Strategy vs Buy & Hold")
plt.legend()
plt.show()

## Conclusion

The strategy is compared against buy-and-hold.

In this case, the strategy underperformed, showing that simple moving average strategies may not always outperform the market.